<a href="https://colab.research.google.com/github/jiya2000/customer-retention-insights-olist/blob/main/notebooks/02_Data_Engineering_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Olist Data Engineering Pipeline Execution

This notebook executes the SQL Data Engineering pipeline defined in the data_engineering/sql/ directory. It uses DuckDB, a high-performance in-memory analytical database, to load the raw CSV files, execute the SQL transformations, and output the final Product Metrics as Pandas DataFrames.

In [1]:
!pip install kagglehub duckdb pandas


1. Download Dataset & Initialize DuckDB

In [2]:

import kagglehub
import duckdb
import pandas as pd
import os

# Download latest version of the Olist dataset
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Path to dataset files:", path)

# Initialize an in-memory DuckDB connection
con = duckdb.connect(database=':memory:')

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


2. Run 01_raw_layer.sql & Load CSVs

Instead of creating empty tables, we will use DuckDB's read_csv_auto to directly load the raw data into the raw schema.

In [3]:

con.execute("CREATE SCHEMA IF NOT EXISTS raw;")

csv_mappings = {
    "olist_customers": "olist_customers_dataset.csv",
    "olist_orders": "olist_orders_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_products": "olist_products_dataset.csv",
    "olist_order_payments": "olist_order_payments_dataset.csv",
    "olist_order_reviews": "olist_order_reviews_dataset.csv",
    "olist_sellers": "olist_sellers_dataset.csv"
}

for table_name, file_name in csv_mappings.items():
    file_path = os.path.join(path, file_name)
    con.execute(f"CREATE TABLE raw.{table_name} AS SELECT * FROM read_csv_auto('{file_path}');")

print("Raw tables loaded successfully.")
con.execute("SHOW TABLES;").df()

Raw tables loaded successfully.


,name


3. Run Data Quality Checks (02_data_quality_checks_raw.sql)

In [4]:
dq_queries = [
    ("Null Purchase Timestamps", "SELECT COUNT(*) AS null_purchase_timestamps FROM raw.olist_orders WHERE order_purchase_timestamp IS NULL;"),
    ("Orphaned Orders", "SELECT COUNT(*) AS orphaned_orders FROM raw.olist_orders o LEFT JOIN raw.olist_customers c ON o.customer_id = c.customer_id WHERE c.customer_id IS NULL;")
]

for desc, query in dq_queries:
    print(f"--- {desc} ---")
    display(con.execute(query).df())
    print("\n")

--- Null Purchase Timestamps ---


,null_purchase_timestamps
0,0




--- Orphaned Orders ---


,orphaned_orders
0,0


4. Run Staging & Modeling (03_cleaning_and_staging.sql & 04_data_modeling.sql)

In [5]:
import urllib.request

def execute_sql_from_github(filename):
    # Fetch the raw SQL file directly from your GitHub repository
    url = f"https://raw.githubusercontent.com/jiya2000/customer-retention-insights-olist/main/data_engineering/sql/{filename}"
    response = urllib.request.urlopen(url)
    sql = response.read().decode('utf-8')

    # DuckDB doesn't like some PostgreSQL specific functions, so we translate them on the fly:
    sql = sql.replace("EXTRACT(EPOCH FROM (order_delivered_customer_date - order_purchase_timestamp))/86400", "date_diff('day', CAST(order_purchase_timestamp AS TIMESTAMP), CAST(order_delivered_customer_date AS TIMESTAMP))")
    sql = sql.replace("EXTRACT(DOW FROM d)", "dayofweek(d)")

    # Fix the TO_CHAR formatting for DuckDB
    sql = sql.replace("TO_CHAR(d, 'YYYYMMDD')", "strftime(d, '%Y%m%d')")
    sql = sql.replace("TO_CHAR(o.order_purchase_timestamp, 'YYYYMMDD')", "strftime(o.order_purchase_timestamp, '%Y%m%d')")

    # Fix the date generation column name for DuckDB
    sql = sql.replace(") AS d;", ") AS t(d);")

    con.execute(sql)

# Run the files directly from GitHub!
execute_sql_from_github('03_cleaning_and_staging.sql')
print("Staging schema created successfully.")

execute_sql_from_github('04_data_modeling.sql')
print("Analytics (Star Schema) created successfully.")


Staging schema created successfully.
Analytics (Star Schema) created successfully.


5. Final Output: Product Metrics (05_product_metrics.sql)

Now we run the business queries on our clean Star Schema to generate actionable product insights.

In [6]:

# 1. Monthly Active Users & GMV
mau_sql = """
SELECT
    d.year,
    d.month,
    COUNT(DISTINCT f.customer_id) AS monthly_active_purchasers,
    SUM(f.total_item_value) AS monthly_gmv
FROM analytics.fact_order_items f
JOIN analytics.dim_date d ON f.purchase_date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month
LIMIT 10;
"""
print("--- Top 10 Months by Active Purchasers & GMV ---")
display(con.execute(mau_sql).df())

--- Top 10 Months by Active Purchasers & GMV ---


,year,month,monthly_active_purchasers,monthly_gmv
0,2016,9,3,354.75
1,2016,10,308,56808.84
2,2016,12,1,19.62
3,2017,1,789,137188.49
4,2017,2,1733,286280.62
5,2017,3,2641,432048.59
6,2017,4,2391,412422.24
7,2017,5,3660,586190.95
8,2017,6,3217,502963.04
9,2017,7,3969,584971.62


In [7]:

# 2. Repeat Purchase Rate
repeat_sql = """
WITH customer_order_ranks AS (
    SELECT
        c.customer_unique_id,
        f.order_id,
        ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY f.purchase_date_key) AS order_sequence_num
    FROM analytics.fact_order_items f
    JOIN analytics.dim_customers c ON f.customer_id = c.customer_id
    GROUP BY c.customer_unique_id, f.order_id, f.purchase_date_key
)
SELECT
    COUNT(DISTINCT CASE WHEN order_sequence_num > 1 THEN customer_unique_id END) * 100.0 /
    COUNT(DISTINCT customer_unique_id) AS repeat_purchase_rate_percentage
FROM customer_order_ranks;
"""
print("--- Repeat Purchase Rate (%) ---")
display(con.execute(repeat_sql).df())

--- Repeat Purchase Rate (%) ---


,repeat_purchase_rate_percentage
0,3.052819
